# FINE TUNING: Los mejores modelos de la etapa anterior fueron: Random Forest, Regresión Logística y SVM

In [3]:
import os
import sys
from hashlib import sha256
from pathlib import Path
from time import perf_counter

import mlflow
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

if not (raiz / "src" / "pipeline.py").is_file():
    raiz = Path(r"C:\USFQ\Maestría\Ciencia_Datos\pset3")

sys.path.insert(0, str(raiz))

from src.datos import cargar_obt
from src.features import construir_features
from src.pipeline import crear_pipeline

SEMILLA = 42

mlflow.set_tracking_uri("http://127.0.0.1:5000")
experimento = mlflow.set_experiment(
    "Sistema de alerta temprana de volatilidad para acciones"
)

print("Raíz del proyecto:", raiz)
print("Experimento:", experimento.name)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Raíz del proyecto: c:\USFQ\Maestría\Ciencia_Datos\pset3
Experimento: Sistema de alerta temprana de volatilidad para acciones


In [4]:
# Selección de tres modelos

import mlflow

ids_candidatos = [
    "b1d407af9d61477c90cf10b99db6df1a",
    "88a42b72b1d74abeb5e513798f264ef0",
    "353404c1476247d0b3fb033570a5aa51",
    "fb70dc2201724331be14e9817aed37f0",
    "86557cb8140b4a7b9af5514ccb3679ad",
]

cliente = mlflow.MlflowClient()
filas = []

for run_id in ids_candidatos:
    run = cliente.get_run(run_id)
    if (
        run.info.status != "FINISHED"
        or run.info.experiment_id != experimento.experiment_id
        or run.data.tags.get("fase") != "candidato"
    ):
        raise ValueError(f"Revisa la corrida candidata {run_id}")

    filas.append({
        "run_id": run_id,
        "modelo": run.data.params["modelo"],
        "auc_default": run.data.metrics["roc_auc_val_media_cv"],
        "std_default": run.data.metrics["roc_auc_val_std_cv"],
    })

ranking = pd.DataFrame(filas).sort_values("auc_default", ascending=False)
assert ranking["modelo"].nunique() == 5, "Selecciona una corrida por candidato"

seleccionados = ranking.head(3)
display(ranking)

,run_id,modelo,auc_default,std_default
1,88a42b72b1d74abeb5e513798f264ef0,RandomForestClassifier,0.721809,0.048433
0,b1d407af9d61477c90cf10b99db6df1a,LogisticRegression,0.709873,0.067944
4,86557cb8140b4a7b9af5514ccb3679ad,SVC,0.699498,0.040125
2,353404c1476247d0b3fb033570a5aa51,HistGradientBoostingClassifier,0.661221,0.045169
3,fb70dc2201724331be14e9817aed37f0,MLPClassifier,0.636036,0.075998


In [5]:
df = cargar_obt()
snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(
        snapshot.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print(datos_corrida)

{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '9e9d8a678f5feb51385ae330e460a538835644c6e2068e9f989fd362ce45d4e8', 'filas_snapshot': 4087}


In [6]:
horizonte = 5
inicio_prueba = pd.Timestamp("2023-07-11")

features = construir_features(df)
vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(horizonte, min_periods=horizonte)
    .std(ddof=1)
    .shift(-horizonte)
    .rename("VOL_FUTURA_5")
)

fin_ventana_target = features.index.to_series().shift(-horizonte)
datos_modelado = features.iloc[60:].join(vol_futura_5).dropna(
    subset=["VOL_FUTURA_5"]
)

entrenamiento = datos_modelado.loc[
    fin_ventana_target.reindex(datos_modelado.index) < inicio_prueba
].copy()

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()
cv_temporal = TimeSeriesSplit(n_splits=5, gap=horizonte)

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

for run_id in ids_candidatos:
    referencia = cliente.get_run(run_id)
    if referencia.data.params.get("sha256_dataset_desarrollo") != hash_desarrollo:
        raise ValueError(f"El dataset no coincide con la corrida {run_id}")

print("Forma de X_desarrollo:", X_desarrollo.shape)

Forma de X_desarrollo: (3212, 18)


In [7]:
def evaluar_candidato(estimador):
    resultados = []

    for pliegue, (idx_train, idx_val) in enumerate(
        cv_temporal.split(X_desarrollo), start=1
    ):
        X_train = X_desarrollo.iloc[idx_train]
        X_val = X_desarrollo.iloc[idx_val]
        vol_train = vol_desarrollo.iloc[idx_train]
        vol_val = vol_desarrollo.iloc[idx_val]

        p80 = vol_train.quantile(0.80)
        y_train = vol_train.gt(p80).astype(int)
        y_val = vol_val.gt(p80).astype(int)

        pipeline = crear_pipeline(clone(estimador), columnas_numericas)
        inicio = perf_counter()
        pipeline.fit(X_train, y_train)
        segundos = perf_counter() - inicio

        auc_train = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:, 1])
        auc_val = roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1])

        resultados.append({
            "pliegue": pliegue,
            "fin_train": X_train.index[-1].date(),
            "inicio_validacion": X_val.index[0].date(),
            "fin_validacion": X_val.index[-1].date(),
            "p80_train": p80,
            "roc_auc_train": auc_train,
            "roc_auc_val": auc_val,
            "brecha_roc_auc": auc_train - auc_val,
            "segundos_ajuste": segundos,
        })

    return pd.DataFrame(resultados)

In [8]:
from scipy.stats import loguniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterSampler
from sklearn.svm import SVC

estimadores = {
    "RandomForestClassifier": RandomForestClassifier(random_state=SEMILLA),
    "LogisticRegression": LogisticRegression(random_state=SEMILLA),
    "SVC": SVC(probability=True, random_state=SEMILLA),
}

espacios = {
    "RandomForestClassifier": {
        "n_estimators": [80, 120, 160],
        "max_depth": [3, 5, 8, 12, None],
        "min_samples_leaf": [2, 5, 10, 20],
        "max_features": ["sqrt", 0.5],
    },
    "LogisticRegression": {
        "C": loguniform(1e-3, 1e2),
    },
    "SVC": {
        "C": loguniform(1e-2, 1e2),
        "gamma": loguniform(1e-4, 1),
    },
}

familias = {
    "RandomForestClassifier": "bagging",
    "LogisticRegression": "lineal_regularizado",
    "SVC": "maquina_vectores_soporte",
}

In [9]:
import subprocess

import mlflow.sklearn
from mlflow.models import infer_signature


def ajustar_modelo(referencia):
    nombre = referencia["modelo"]
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()

    configuraciones = list(ParameterSampler(
        espacios[nombre], n_iter=30, random_state=SEMILLA
    ))

    tags = {
        "fase": "tuning",
        "familia": familias[nombre],
        "git_commit": commit,
    }
    datos_tuning = {
        **datos_corrida,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "run_id_default": referencia["run_id"],
        "semilla_busqueda": SEMILLA,
        "n_pliegues": cv_temporal.n_splits,
        "gap": cv_temporal.gap,
        "cuantil_target": 0.80,
    }
    filas = []

    with mlflow.start_run(
        experiment_id=experimento.experiment_id,
        run_name=f"tuning_{nombre}",
    ) as padre:
        mlflow.set_tags(tags)
        mlflow.log_params({**datos_tuning, "n_configuraciones": len(configuraciones)})

        for numero, parametros in enumerate(configuraciones, start=1):
            candidato = clone(estimadores[nombre]).set_params(**parametros)

            with mlflow.start_run(
                run_name=f"{nombre}_{numero:02d}", nested=True
            ) as hija:
                mlflow.set_tags(tags)
                mlflow.log_params({
                    **datos_tuning,
                    "configuracion": numero,
                    "semilla_modelo": SEMILLA,
                    **{f"modelo__{k}": v for k, v in candidato.get_params().items()},
                })

                resultados = evaluar_candidato(candidato)
                metricas = {
                    "roc_auc_train_media_cv": float(resultados["roc_auc_train"].mean()),
                    "roc_auc_val_media_cv": float(resultados["roc_auc_val"].mean()),
                    "roc_auc_val_std_cv": float(resultados["roc_auc_val"].std(ddof=1)),
                    "brecha_roc_auc_cv": float(resultados["brecha_roc_auc"].mean()),
                    "segundos_ajuste_cv_total": float(resultados["segundos_ajuste"].sum()),
                }

                mlflow.log_metrics(metricas)
                mlflow.log_text(
                    resultados.to_csv(index=False), "resultados_por_pliegue.csv"
                )
                filas.append({
                    "configuracion": numero,
                    "run_id": hija.info.run_id,
                    **parametros,
                    **metricas,
                })

            print(
                f"{nombre} {numero}/{len(configuraciones)} · "
                f"AUC {metricas['roc_auc_val_media_cv']:.4f}",
                flush=True,
            )

        resumen = pd.DataFrame(filas).sort_values(
            "roc_auc_val_media_cv", ascending=False, kind="stable"
        )
        mejor = resumen.iloc[0]
        parametros_mejor = configuraciones[int(mejor["configuracion"]) - 1]

        auc_default = float(referencia["auc_default"])
        auc_ajustado = float(mejor["roc_auc_val_media_cv"])
        std_ajustado = float(mejor["roc_auc_val_std_cv"])
        mejora = auc_ajustado - auc_default
        mejora_segun_pset = mejora > 0 and mejora >= std_ajustado

        mlflow.log_metrics({
            "roc_auc_default": auc_default,
            "roc_auc_default_std_cv": float(referencia["std_default"]),
            "roc_auc_val_media_cv": auc_ajustado,
            "roc_auc_val_std_cv": std_ajustado,
            "mejora_auc": mejora,
        })
        mlflow.set_tags({
            "mejor_run_hijo": mejor["run_id"],
            "mejora_segun_pset": str(mejora_segun_pset).lower(),
        })
        mlflow.log_dict(parametros_mejor, "mejores_parametros.json")
        mlflow.log_text(resumen.to_csv(index=False), "resumen_tuning.csv")

        p80_desarrollo = float(vol_desarrollo.quantile(0.80))
        y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)
        mlflow.log_param("p80_desarrollo", p80_desarrollo)

        mejor_estimador = clone(estimadores[nombre]).set_params(**parametros_mejor)
        pipeline = crear_pipeline(mejor_estimador, columnas_numericas)

        inicio = perf_counter()
        pipeline.fit(X_desarrollo, y_desarrollo)
        mlflow.log_metric("segundos_reajuste_desarrollo", perf_counter() - inicio)

        ejemplo = X_desarrollo.head(5).copy()
        modelo_registrado = mlflow.sklearn.log_model(
            pipeline,
            name=f"pipeline_{nombre}_ajustado",
            signature=infer_signature(ejemplo, pipeline.predict_proba(ejemplo)),
            input_example=ejemplo,
            pyfunc_predict_fn="predict_proba",
        )

    print(f"Run padre: {padre.info.run_id}")
    print(f"URI del modelo: {modelo_registrado.model_uri}")
    print(f"Default: {auc_default:.4f} ± {referencia['std_default']:.4f}")
    print(f"Ajustado: {auc_ajustado:.4f} ± {std_ajustado:.4f}")
    print(f"Mejora: {mejora:+.4f} | Cumple criterio PSet: {mejora_segun_pset}")
    display(resumen.head())
    return resumen

In [10]:
tuning_bosque = ajustar_modelo(seleccionados.iloc[0])

🏃 View run RandomForestClassifier_01 at: http://127.0.0.1:5000/#/experiments/1/runs/e01aae6dc6f14706a62b0eb0b8d1d995
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
RandomForestClassifier 1/30 · AUC 0.7295
🏃 View run RandomForestClassifier_02 at: http://127.0.0.1:5000/#/experiments/1/runs/b4afd99dbfe54c53aaafe5062b52a4ff
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
RandomForestClassifier 2/30 · AUC 0.7330
🏃 View run RandomForestClassifier_03 at: http://127.0.0.1:5000/#/experiments/1/runs/385278922b5a4f989f5dc352bf5b8cec
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
RandomForestClassifier 3/30 · AUC 0.7459
🏃 View run RandomForestClassifier_04 at: http://127.0.0.1:5000/#/experiments/1/runs/bd6ec80479b64ee5bfdd732f15153479
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
RandomForestClassifier 4/30 · AUC 0.7371
🏃 View run RandomForestClassifier_05 at: http://127.0.0.1:5000/#/experiments/1/runs/1b40193daacb43c091e157a1acba7181
🧪 View expe

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 19:13:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run tuning_RandomForestClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/8e36ee19a4ac415cb8fd7baef1a64261
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run padre: 8e36ee19a4ac415cb8fd7baef1a64261
URI del modelo: models:/m-94dd0f01b41e4cb8b2c990b2ee0a3739
Default: 0.7218 ± 0.0484
Ajustado: 0.7478 ± 0.0625
Mejora: +0.0260 | Cumple criterio PSet: False


,configuracion,run_id,n_estimators,min_samples_leaf,max_features,max_depth,roc_auc_train_media_cv,roc_auc_val_media_cv,roc_auc_val_std_cv,brecha_roc_auc_cv,segundos_ajuste_cv_total
12,13,eef4d9cd4dbe4cd2ac11822bb43fd57d,160,20,sqrt,3.0,0.793910,0.747780,0.062476,0.046130,6.134879
7,8,847548acbe69422996aff28ac046360e,120,20,sqrt,3.0,0.795205,0.747655,0.061569,0.047549,3.332307
17,18,7e25e66ec24a44b4b5f33071595891b2,80,2,sqrt,3.0,0.804985,0.746151,0.062439,0.058834,5.989662
2,3,385278922b5a4f989f5dc352bf5b8cec,120,5,sqrt,3.0,0.806670,0.745927,0.065670,0.060744,4.081892
22,23,fddd843e012e416390db4ce488d64f13,120,10,sqrt,5.0,0.858756,0.742685,0.064605,0.116071,7.040893


In [11]:
tuning_logistica = ajustar_modelo(seleccionados.iloc[1])

🏃 View run LogisticRegression_01 at: http://127.0.0.1:5000/#/experiments/1/runs/5f80015ef25b47af8b8ffefb6e0627ef
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
LogisticRegression 1/30 · AUC 0.7245
🏃 View run LogisticRegression_02 at: http://127.0.0.1:5000/#/experiments/1/runs/c552e50bbd9f4ea1a768ddf14cf43b32
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
LogisticRegression 2/30 · AUC 0.7045
🏃 View run LogisticRegression_03 at: http://127.0.0.1:5000/#/experiments/1/runs/bd5d2dbf16fd4bfc8b143a85e97abb64
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
LogisticRegression 3/30 · AUC 0.7058
🏃 View run LogisticRegression_04 at: http://127.0.0.1:5000/#/experiments/1/runs/31794766226d452d828c427598104fb9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
LogisticRegression 4/30 · AUC 0.7099
🏃 View run LogisticRegression_05 at: http://127.0.0.1:5000/#/experiments/1/runs/a77ef119928c4578ba54f7797a2ece45
🧪 View experiment at: http://127.0.0.1:5000/#/e

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 19:19:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run tuning_LogisticRegression at: http://127.0.0.1:5000/#/experiments/1/runs/38caf344f8424927a1ee39f6aabbcdc6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run padre: 38caf344f8424927a1ee39f6aabbcdc6
URI del modelo: models:/m-caec10a03dc140328d4676d90bd14bcf
Default: 0.7099 ± 0.0679
Ajustado: 0.7353 ± 0.0556
Mejora: +0.0254 | Cumple criterio PSet: False


,configuracion,run_id,C,roc_auc_train_media_cv,roc_auc_val_media_cv,roc_auc_val_std_cv,brecha_roc_auc_cv,segundos_ajuste_cv_total
29,30,adc670a1c90441b9b4bc364e8d2884d2,0.001707,0.711539,0.735320,0.055646,-0.023781,0.242256
10,11,bf0206e7aa14453599df8f01e181ad40,0.001267,0.708872,0.735302,0.055660,-0.026430,0.136637
6,7,cf8453b04a2e42969dfdf5cf56f782b3,0.001952,0.712820,0.735266,0.055559,-0.022446,0.260124
21,22,6f58cf4c3232475eb392b3a9d27cbfdd,0.004983,0.725083,0.735087,0.055855,-0.010004,0.182032
5,6,8a4894eb57c1419fb7cbd191c2aacd04,0.006025,0.728233,0.734953,0.055810,-0.006720,0.138027


In [12]:
tuning_svm = ajustar_modelo(seleccionados.iloc[2])

🏃 View run SVC_01 at: http://127.0.0.1:5000/#/experiments/1/runs/ad2b51477cc2475fa77b0626456640f1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
SVC 1/30 · AUC 0.6581
🏃 View run SVC_02 at: http://127.0.0.1:5000/#/experiments/1/runs/b6da69f266c44beda990c7eee75f9f00
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
SVC 2/30 · AUC 0.6706
🏃 View run SVC_03 at: http://127.0.0.1:5000/#/experiments/1/runs/c1dccaeaa67a4ea7b4b7eb9a2dc2f955
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
SVC 3/30 · AUC 0.7033
🏃 View run SVC_04 at: http://127.0.0.1:5000/#/experiments/1/runs/8b39141de9824a21b35b35f31f2413d2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
SVC 4/30 · AUC 0.6905
🏃 View run SVC_05 at: http://127.0.0.1:5000/#/experiments/1/runs/47fb821b87d04068a29343525fc84774
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
SVC 5/30 · AUC 0.6888
🏃 View run SVC_06 at: http://127.0.0.1:5000/#/experiments/1/runs/00bc514b6ecb4c11badf9f446ef13949
🧪 

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 19:29:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run tuning_SVC at: http://127.0.0.1:5000/#/experiments/1/runs/5b22d6fe28cc410aa12e4221f623147a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run padre: 5b22d6fe28cc410aa12e4221f623147a
URI del modelo: models:/m-4a1708bb281645798def9b1107f6a0c9
Default: 0.6995 ± 0.0401
Ajustado: 0.7299 ± 0.0553
Mejora: +0.0304 | Cumple criterio PSet: False


,configuracion,run_id,C,gamma,roc_auc_train_media_cv,roc_auc_val_media_cv,roc_auc_val_std_cv,brecha_roc_auc_cv,segundos_ajuste_cv_total
24,25,370afcd9a1a34f3f9e0d90a0cafdb198,1.537592,0.000549,0.771239,0.729891,0.055308,0.041348,5.863168
7,8,46cb45e1d950471b87e5157fb20786f2,0.053370,0.000542,0.748179,0.729783,0.069124,0.018396,4.614279
10,11,48a3efcb106a456dabc1086b13332153,2.801635,0.000361,0.772962,0.728921,0.053429,0.044041,6.199717
15,16,9eb6563d1c0b484ba1174265fc129951,2.692647,0.000481,0.775790,0.727095,0.053730,0.048695,6.702369
9,10,11ae2792892e451db51d4a1ad8e7c453,0.534294,0.001462,0.777243,0.722364,0.060058,0.054880,5.703257
